# 🎵 VIDEO-FREE · 1. Canción completa con voz (ACE-Step 1.5)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/1_cancion_ia.ipynb)

Toma la **letra, el estilo, el BPM y la tonalidad** de tu proyecto y genera una canción completa **con voz cantando en español** e instrumentos.
Después separa la voz del instrumental, sincroniza la letra palabra por palabra con la voz y guarda todo en tu repo para que el portal lo use.

- **Modelo:** ACE-Step 1.5 XL Turbo (licencia MIT). Separación: Demucs (MIT). Tiempos de la letra: Whisper con faster-whisper (MIT).
- **GPU:** sirve la T4 gratuita (más lenta). Con L4 o A100 va mucho más rápido.
- **Tiempo aproximado:** 5–10 min la primera vez (descarga de modelos, ~12 GB) y 1–3 min por canción después.
- La voz que sale es **una voz de IA**, no la tuya. Para que cante con tu timbre usa después el cuaderno **2 · Mi voz**.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar (tarda unos minutos la primera vez)
!pip install -q -U "diffusers>=0.43" transformers accelerate soundfile demucs faster-whisper
print("Listo")

In [ ]:
#@title 3 · Elegir proyecto y estilo
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
#@markdown Describe el estilo en inglés (el modelo lo entiende mejor). Déjalo vacío para usar el estilo del proyecto.
ESTILO = ""  #@param {type:"string"}
VOZ = "male vocals"  #@param ["male vocals", "female vocals", "male and female vocals"]
DURACION = 0  #@param {type:"number"}
#@markdown `DURACION = 0` usa la duración del proyecto (mínimo 10 s, máximo 600 s).
VARIANTES = 2  #@param {type:"slider", min:1, max:4, step:1}
SEMILLA = 7  #@param {type:"integer"}

p = comun.proyecto(PROYECTO)
ESTILOS = {
    "reggaeton": "reggaeton, dembow rhythm, latin urban, deep 808 bass, emotional, melodic, modern production",
    "balada": "latin pop ballad, piano and acoustic guitar, strings, emotional, heartfelt",
    "trap": "latin trap, 808 bass, hi-hat rolls, dark atmosphere, melodic",
}
estilo = ESTILO.strip() or ESTILOS.get((p.get("arreglo") or {}).get("estilo", "reggaeton"), ESTILOS["reggaeton"])
prompt = f"{estilo}, {VOZ}, spanish lyrics, high quality studio mix"
letra = comun.letra_con_secciones(p)
duracion = float(DURACION or p.get("duracion") or 60)
duracion = max(10.0, min(600.0, duracion))
bpm = int(p.get("bpm") or 95)
tono = comun.tonalidad_en_ingles(p)
print("Estilo:", prompt); print("BPM:", bpm, "· Tonalidad:", tono or "la elige el modelo", "· Duración:", duracion, "s")
print("\nLetra (puedes editarla en la celda siguiente):\n"); print(letra)

In [ ]:
#@title 4 · (Opcional) Editar la letra antes de generar
#@markdown Si quieres otra estructura, escribe la letra aquí con etiquetas como `[verse]`, `[chorus]`, `[bridge]`. Déjalo vacío para usar la del proyecto.
LETRA_PROPIA = """"""
if LETRA_PROPIA.strip():
    letra = LETRA_PROPIA.strip()
    print("Usaré tu letra editada.")
else:
    print("Usaré la letra del proyecto.")

In [ ]:
#@title 5 · Generar la canción
import torch, soundfile as sf
from diffusers import AceStepPipeline

capacidad = torch.cuda.get_device_capability()[0]
dtype = torch.bfloat16 if capacidad >= 8 else torch.float16  # la T4 no tiene bfloat16 nativo
if "pipe" not in globals():
    # Los pesos (~11 GB) se cargan directo a la GPU, sin pasar por la RAM del sistema (12 GB en Colab gratis)
    pipe = AceStepPipeline.from_pretrained("ACE-Step/acestep-v15-xl-turbo-diffusers", torch_dtype=dtype, device_map="cuda", low_cpu_mem_usage=True)
    pipe.vae.enable_tiling()
    print(f"Modelo en la GPU: {torch.cuda.memory_allocated() / 2**30:.1f} GB usados")

def generar(semilla):
    return pipe(prompt=prompt, lyrics=letra, audio_duration=duracion, vocal_language="es", bpm=bpm, keyscale=tono,
                timesignature="4", generator=torch.Generator(device="cuda").manual_seed(semilla)).audios[0]

variantes = []
for i in range(VARIANTES):
    audio = generar(SEMILLA + i)
    if not torch.isfinite(audio).all() or audio.abs().max() < 1e-4:  # en algunas T4 la media precisión falla
        print("La variante salió vacía en media precisión; repito en bfloat16.")
        pipe.to(torch.bfloat16)
        audio = generar(SEMILLA + i)
    ruta = comun.TRABAJO / f"variante_{i + 1}.wav"
    sf.write(ruta, audio.T.cpu().float().numpy(), pipe.sample_rate)
    variantes.append(ruta)
    print(f"Variante {i + 1}:"); comun.escuchar(ruta)

In [ ]:
#@title 6 · Elegir la variante, separar la voz y sincronizar la letra
ELEGIDA = 1  #@param {type:"integer"}
cancion = variantes[ELEGIDA - 1]
# Libera la GPU: el modelo de la canción ocupa ~11 GB y Whisper necesita espacio
import gc, torch
if "pipe" in globals():
    del pipe
gc.collect(); torch.cuda.empty_cache()
voz, instrumental = comun.separar_voz(cancion)
print("Voz sola:"); comun.escuchar(voz)

# Tiempos de cada palabra: Whisper escucha la voz y se alinean con la letra escrita
lineas = [l for l in letra.splitlines() if l.strip() and not l.strip().startswith("[")]
palabras = comun.tiempos_de_palabras(voz, "es")
letra_sincronizada = comun.alinear_letra(palabras, lineas)
for l in letra_sincronizada[:6]:
    print(f"{l['inicio']:6.2f}–{l['fin']:6.2f}  {l['texto']}")

In [ ]:
#@title 7 · Guardar en tu repo y en Drive
nombre = comun.slug(p.get("titulo", "cancion"))
archivos = {
    "cancion": comun.a_mp3(cancion, comun.TRABAJO / f"{nombre}_ia.mp3"),
    "voz": comun.a_mp3(voz, comun.TRABAJO / f"{nombre}_voz_ia.mp3", lufs=-16),
    "instrumental": comun.a_mp3(instrumental, comun.TRABAJO / f"{nombre}_instrumental_ia.mp3"),
}
# Los WAV quedan en Drive para el cuaderno 2 (convertir a tu voz)
comun.a_drive(voz, f"{nombre}_voz_ia.wav"); comun.a_drive(instrumental, f"{nombre}_instrumental_ia.wav"); comun.a_drive(archivos["cancion"])
for clave, ruta in archivos.items():
    comun.guardar(f"medios/audio/{ruta.name}", ruta.read_bytes(), f"Canción IA: {clave} de {p.get('titulo')}")

p["audio_anterior"] = p.get("audio")
p["audio"] = f"medios/audio/{archivos['cancion'].name}"
p["duracion"] = round(comun.duracion(archivos["cancion"]), 1)
p["letra"] = letra_sincronizada
p["ia"] = {**p.get("ia", {}), "cancion": {"modelo": "ACE-Step 1.5 XL Turbo", "estilo": prompt, "semilla": SEMILLA + ELEGIDA - 1,
           "voz": f"medios/audio/{archivos['voz'].name}", "instrumental": f"medios/audio/{archivos['instrumental'].name}"}}
if p.get("canto"):
    p["canto"]["activo"] = False  # ya canta la voz de la canción
comun.guardar_proyecto(PROYECTO, p, f"Proyecto: canción IA para {p.get('titulo')}")
print("\nListo. En el portal pulsa «Recargar proyecto» (pestaña IA) para oír la canción con la letra sincronizada.")